# Stage 3 -- Theme Allocation: Combined Means

## Input
- `Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_combined_means.parquet` -- schema read only, to obtain exact feature column names
- `Data/Data_Collection/Final/Stage_3_Model_Ready/combined_means_factor_inventory.csv` -- source/description metadata for each feature

## Purpose
Assigns every feature in the combined means table to a hierarchical theme/subtheme taxonomy. The output CSV annotates each feature with a `theme_id`, `theme_name`, `subtheme_id`, and `subtheme_name`, alongside the existing source/description metadata. This taxonomy is used for feature grouping in model analysis, ablation studies, and interpretation.

---

## Structure

### Step 1: Read Feature Column Names
The parquet schema is read without loading data. `date`, `target_daily_return`, and `target_monthly_return` are excluded, leaving the full list of features.

### Step 2: Define Theme/Subtheme Mapping
A hand-crafted mapping assigns each feature to exactly one `(theme_id, theme_name, subtheme_id, subtheme_name)` tuple. Features are assigned in bulk using a helper function `assign(factors, ...)`. The 14 themes and their subthemes are:

| # | Theme | Subthemes |
|---|---|---|
| 1 | Liquidity & Market Quality | CRSP bid-ask, TAQ effective spreads, quoted spreads & intraday NBBO, best-level depth, intraday depth profile, depth imbalance, price impact, realized spreads & Kyle lambda, market efficiency & execution, monthly liquidity |
| 2 | Order Flow & Participation | Aggregate/institutional/retail buy-sell balance, auction imbalances & intraday drift, Lee-Ready/institutional/retail classified volume, turnover & volume dynamics, venue distribution, trade size & timing |
| 3 | Volatility & Options | Stock IV levels, vol surface shape, VRP & IV dynamics, Greeks & dealer positioning, OI distribution, put-call sentiment & parity, stock realized vol, TAQ intraday vol, VIX & market realized vol, VIX futures & term structure, CBOE SKEW, CFTC VIX futures positioning |
| 4 | Momentum & Reversal | Daily returns & intraday direction, short-term momentum, style factor returns, factor momentum, medium-term momentum, reversal & anchoring, seasonal momentum, off-season momentum, momentum dynamics |
| 5 | Valuation | Earnings & enterprise multiples, book value & intangible-adjusted, sales/cash flow/revenue yields, payout & duration, forward valuation from analysts |
| 6 | Profitability & Earnings Quality | Profitability measures, accruals & NOA, earnings dynamics, tax & special items |
| 7 | Investment & Corporate Structure | Asset growth & capex, equity issuance & buybacks, debt & leverage, balance sheet changes, financial assets & cash, growth & efficiency, industry structure & firm characteristics, corporate events |
| 8 | Analyst Expectations & Sentiment | PT consensus/disagreement/revisions, recommendation consensus/dynamics/smoothed, revenue estimates/revisions/cross-horizon, cross-signal alignment, short interest & institutional positioning, retail sentiment survey |
| 9 | Interest Rates & Monetary Policy | Yield curve level/shape, inflation expectations & real rates, policy rates, yield dynamics, yield regime, Fed balance sheet & banking, bond returns, T-bill returns & bond momentum, bond term premium, money supply |
| 10 | Credit Conditions | IG spreads, HY spreads, credit quality differentials, credit dynamics & volatility |
| 11 | Cross-Sectional Risk Profile | Systematic risk exposure & dynamics, idiosyncratic volatility, realized vol & cash flow risk, coskewness & tail risk |
| 12 | Macroeconomic Fundamentals | Employment & labour, inflation, inflation dynamics, production & manufacturing, orders & inventories, consumer & spending, consumer dynamics, housing, trade & external, surveys & leading indicators |
| 13 | Global Markets, FX & Commodities | Developed/EM FX, dollar indices, world equity indices, global equity dynamics, energy commodities, monthly commodities, cross-asset regime |
| 14 | Calendar & Regime | Calendar features, regime indicators |

### Step 3: Validate
Three checks are run:

- **Orphans** (features in data but not in mapping): must be zero for validation to pass
- **Extras** (features in mapping but not in data): must be zero
- **Duplicates** (features assigned more than once): must be zero

A theme summary table is printed showing the number of features and subthemes per theme. Validation passes only if all three checks return zero.

### Step 4: Build and Save CSV
The mapping is joined with the existing `combined_means_factor_inventory.csv` to add `base_factor`, `frequency`, `panel`, `source`, and `description` columns alongside the theme assignment. The result is one row per feature.

---

## Key Notes
- The mapping is entirely hand-crafted -- there is no automated assignment. Every feature is explicitly listed under its subtheme.
- The `assign()` helper warns on duplicates at assignment time (prints a `⚠ DUPLICATE` message), providing early detection before the formal validation step.
- Theme 14 (Calendar & Regime) contains the binary and regime indicator features that were not z-scored in Stage 3.
- Monthly features (prefixed `monthly_`) are assigned to the same thematic groups as their daily counterparts where applicable (e.g., `monthly_BidAskSpread` → Theme 1 subtheme 1.10, `monthly_Mom12m` → Theme 4 subtheme 4.5).

## Output
`Data/Data_Collection/Final/Stage_3_Model_Ready/themes/combined_means_theme_assignment.csv` -- one row per feature, columns: `column`, `theme_id`, `theme_name`, `subtheme_id`, `subtheme_name`, `base_factor`, `frequency`, `panel`, `source`, `description`

In [2]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from collections import Counter

BASE = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: READ ACTUAL COLUMN NAMES FROM COMBINED MEANS TABLE
# ═══════════════════════════════════════════════════════════════════════════════

schema = pq.read_schema(BASE / 'model_market_combined_means.parquet')
all_cols = [f.name for f in schema]
features = [c for c in all_cols if c not in ['date', 'target_daily_return', 'target_monthly_return']]
print(f"Features in combined means table: {len(features)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: DEFINE COMPLETE FACTOR → (THEME, SUBTHEME) MAPPING
# ═══════════════════════════════════════════════════════════════════════════════

# Format: factor_name: (theme_id, theme_name, subtheme_id, subtheme_name)

mapping = {}

def assign(factors, theme_id, theme_name, subtheme_id, subtheme_name):
    for f in factors:
        if f in mapping:
            print(f"  ⚠ DUPLICATE: {f} already assigned to {mapping[f][2]}, now also {subtheme_id}")
        mapping[f] = (theme_id, theme_name, subtheme_id, subtheme_name)

# ── THEME 1: Liquidity & Market Quality ──────────────────────────────────────

T, TN = 1, "Liquidity & Market Quality"

assign([
    'bid_ask_spread', 'spread_5d_mean', 'spread_20d_mean',
    'spread_rel_5d', 'spread_rel_20d',
], T, TN, '1.1', 'CRSP Bid-Ask Spread & Dynamics')

assign([
    'effectivespread_dollar_ave', 'effectivespread_dollar_dw', 'effectivespread_dollar_sw',
    'effectivespread_percent_ave', 'effectivespread_percent_dw', 'effectivespread_percent_sw',
    'effspread_pct_rel_5d',
], T, TN, '1.2', 'TAQ Effective Spreads')

assign([
    'quotedspread_dollar_tw', 'quotedspread_percent_tw',
    'nbbo_spread_open', 'nbbo_spread_1pm', 'nbbo_spread_4pm', 'nbbo_spread_close',
], T, TN, '1.3', 'Quoted Spreads & Intraday NBBO')

assign([
    'bestbiddepth_dollar_tw_to_cap', 'bestofrdepth_dollar_tw_to_cap',
    'bestbiddepth_share_tw_to_shrout', 'bestofrdepth_share_tw_to_shrout',
], T, TN, '1.4', 'Best-Level Depth')

assign([
    'nbbqty_after_open_to_shrout', 'nbbqty_1pm_to_shrout',
    'nbbqty_4pm_to_shrout', 'nbbqty_before_close_to_shrout',
    'nboqty_after_open_to_shrout', 'nboqty_1pm_to_shrout',
    'nboqty_4pm_to_shrout', 'nboqty_before_close_to_shrout',
], T, TN, '1.5', 'Intraday Depth Profile')

assign([
    'depth_imbalance', 'depth_imbalance_5d_mean', 'depth_imbalance_chg_1d',
    'bid_depth_rel_5d', 'offer_depth_rel_5d',
], T, TN, '1.6', 'Depth Imbalance & Dynamics')

assign([
    'dollarpriceimpact_lr_ave', 'dollarpriceimpact_lr_dw', 'dollarpriceimpact_lr_sw',
    'percentpriceimpact_lr_ave', 'percentpriceimpact_lr_dw', 'percentpriceimpact_lr_sw',
    'priceimpact_rel_5d',
], T, TN, '1.7', 'Price Impact')

assign([
    'dollarrealizedspread_lr_ave', 'dollarrealizedspread_lr_dw', 'dollarrealizedspread_lr_sw',
    'percentrealizedspread_lr_ave', 'percentrealizedspread_lr_dw', 'percentrealizedspread_lr_sw',
    'realspread_rel_5d',
    'tsignsqrtdvol1', 'tsignsqrtdvol2',
], T, TN, '1.8', 'Realized Spreads & Kyle Lambda')

assign([
    'var_ratio1', 'var_ratio2', 'var_ratio3', 'var_ratio4', 'var_ratio5',
    'n_obs', 'vwap_deviation_m',
], T, TN, '1.9', 'Market Efficiency & Execution')

assign([
    'monthly_BidAskSpread', 'monthly_DolVol', 'monthly_bidask_3m_avg',
    'monthly_ps_level', 'monthly_ps_innov',
], T, TN, '1.10', 'Monthly Liquidity')

# ── THEME 2: Order Flow & Participation ──────────────────────────────────────

T, TN = 2, "Order Flow & Participation"

assign([
    'bs_ratio_num', 'bs_ratio_vol', 'buysell_premium_lr',
    'bs_ratio_vol_5d_mean', 'bs_ratio_vol_chg_1d',
], T, TN, '2.1', 'Aggregate Buy-Sell Balance')

assign([
    'bs_ratio_inst50k_num', 'bs_ratio_inst50k_vol', 'buysell_premium_inst50k',
    'bs_ratio_inst_5d_mean', 'bs_ratio_inst_chg_5d',
], T, TN, '2.2', 'Institutional Buy-Sell Balance')

assign([
    'bs_ratio_retail_num', 'bs_ratio_retail_vol', 'buysell_premium_retail',
    'retail_dv_share', 'retail_vol_5d_mean',
], T, TN, '2.3', 'Retail Buy-Sell Balance')

assign([
    'close_vs_mid', 'open_vs_mid',
    'intraday_drift', 'midday_drift', 'morning_drift',
], T, TN, '2.4', 'Auction Imbalances & Intraday Drift')

assign([
    'buy_dv_lr_to_cap', 'sell_dv_lr_to_cap', 'total_dv_lr_to_cap',
    'buyvol_lr_to_shrout', 'sellvol_lr_to_shrout',
    'buynumtrades_lr_pct', 'sellnumtrades_lr_pct',
    'n_outside_nbbo_trade_pct',
], T, TN, '2.5', 'Lee-Ready Classified Volume')

assign([
    'buy_dv_inst50k_to_cap', 'sell_dv_inst50k_to_cap', 'total_dv_inst50k_to_cap',
    'buyvol_inst50k_to_shrout', 'sellvol_inst50k_to_shrout', 'total_vol_inst50k_to_shrout',
    'buynumtrades_inst50k_pct', 'sellnumtrades_inst50k_pct', 'total_trade_inst50k_pct',
], T, TN, '2.6', 'Institutional Volume')

assign([
    'buy_dv_retail_to_cap', 'sell_dv_retail_to_cap', 'total_dv_retail_to_cap',
    'buyvol_retail_to_shrout', 'sellvol_retail_to_shrout', 'total_vol_retail_to_shrout',
    'buynumtrades_retail_pct', 'sellnumtrades_retail_pct', 'total_trade_retail_pct',
], T, TN, '2.7', 'Retail Volume')

assign([
    'turnover', 'dvol_to_cap',
    'turnover_5d_mean', 'turnover_20d_mean',
    'turnover_rel_5d', 'turnover_rel_20d',
    'total_vol_to_shrout', 'total_vol_m_to_shrout',
    'volume_return_corr_20d',
    'monthly_VolumeTrend', 'monthly_volumetrend_chg_1m',
], T, TN, '2.8', 'Turnover & Volume Dynamics')

assign([
    'total_dollar_a_to_cap', 'total_dollar_b_to_cap', 'total_dollar_m_to_cap',
    'total_vol_a_to_shrout', 'total_vol_b_to_shrout',
    'total_n_trades_a_pct', 'total_n_trades_b_pct', 'total_n_trades_m_pct',
    'hindex',
], T, TN, '2.9', 'Venue Distribution')

assign([
    'csize_to_shrout', 'osize_to_shrout',
    'size_1pm_to_shrout', 'size_4pm_to_shrout',
], T, TN, '2.10', 'Trade Size & Timing')

# ── THEME 3: Volatility & Options ────────────────────────────────────────────

T, TN = 3, "Volatility & Options"

assign([
    'iv_catm', 'iv_PATM', 'iv_POTM',
    'iv_91d_atm', 'iv_30d_call25', 'iv_30d_put25',
], T, TN, '3.1', 'Stock Implied Volatility Levels')

assign([
    'Skew_OTM', 'vol_term_structure', 'vol_smile',
    'stock_skew_chg_5d', 'vol_term_chg_5d',
], T, TN, '3.2', 'Volatility Surface Shape')

assign([
    'hvol', 'rv_30d', 'vrp_rv', 'vrp_hvol', 'vrp_rv_chg_5d',
    'iv_catm_chg_1d', 'iv_catm_chg_5d', 'iv_catm_rel_20d',
], T, TN, '3.3', 'VRP & IV Dynamics')

assign([
    'oi_wt_delta', 'oi_wt_gamma', 'oi_wt_vega', 'oi_wt_theta',
    'gex_norm', 'dex_norm', 'delta_dollar_volume_norm', 'gex_norm_chg_5d',
], T, TN, '3.4', 'Greeks Exposure & Dealer Positioning')

assign([
    'total_oi_norm', 'total_volume_norm', 'total_oi_norm_chg_5d',
    'sumOI_c_money1_pct', 'sumOI_c_money2_pct', 'sumOI_c_money3_pct',
    'sumOI_p_money1_pct', 'sumOI_p_money2_pct', 'sumOI_p_money3_pct',
], T, TN, '3.5', 'Open Interest Distribution')

assign([
    'PC_Ratio', 'pc_ratio_chg_5d', 'pc_ratio_rel_20d',
    'Parity_VSpread', 'nopt_Parity',
], T, TN, '3.6', 'Put-Call Sentiment & Parity')

assign([
    'ret_vol_5d', 'ret_vol_20d', 'ret_vol_ratio',
    'ret_max_5d', 'ret_min_5d',
    'intraday_range', 'intraday_range_rel_5d', 'intraday_range_5d_mean',
], T, TN, '3.7', 'Stock Realized Volatility')

assign([
    'ivol_q', 'ivol_t',
    'venue_range_m', 'venue_range_a', 'venue_range_b',
], T, TN, '3.8', 'TAQ Intraday Volatility')

assign([
    'vix', 'vxn', 'vxd', 'vix_vxn_ratio', 'vix_vxd_ratio',
    'vix_chg_1d', 'vix_chg_5d', 'vix_pct_chg_1d',
    'vix_vs_ma20', 'vix_vs_ma50',
    'vix_intraday_range', 'vix_overnight_gap',
    'vxn_intraday_range', 'vxn_overnight_gap',
    'vxd_intraday_range', 'vxd_overnight_gap',
    'mktrf_vol_5d', 'mktrf_vol_20d', 'mktrf_vol_ratio',
], T, TN, '3.9', 'VIX & Market Realized Volatility')

assign([
    'vix_fut_front', 'vix_fut_second',
    'vix_term_spread', 'vix_term_ratio', 'vix_futures_basis',
    'vix_fut_ret_1d', 'vix_term_spread_5d_chg',
    'vix_fut_volume', 'vix_fut_oi',
], T, TN, '3.10', 'VIX Futures & Term Structure')

assign([
    'skew', 'skew_excess', 'skew_pctile_252d',
    'skew_chg_5d', 'skew_ma20', 'skew_vs_ma20',
], T, TN, '3.11', 'CBOE SKEW')

assign([
    'lev_long', 'lev_short', 'lev_spread',
    'lev_net', 'lev_net_pct', 'lev_net_chg', 'lev_am_ratio',
    'am_long', 'am_short', 'am_spread',
    'am_net', 'am_net_pct', 'am_net_chg',
    'dealer_long', 'dealer_short', 'dealer_spread',
    'dealer_net', 'dealer_net_pct',
    'other_long', 'other_short', 'other_spread',
    'open_interest',
], T, TN, '3.12', 'CFTC VIX Futures Positioning')

# ── THEME 4: Momentum & Reversal ─────────────────────────────────────────────

T, TN = 4, "Momentum & Reversal"

assign([
    'dlyretx', 'dlyreti', 'open_to_close_ret',
    'ret_mkt_m', 'open_to_close_ret_5d_mean',
], T, TN, '4.1', 'Daily Returns & Intraday Direction')

assign([
    'ret_cum_5d', 'ret_cum_20d', 'n30_pos', 'n5_pos',
], T, TN, '4.2', 'Short-Term Momentum')

assign([
    'mktrf', 'smb', 'hml', 'rmw', 'cma', 'umd',
], T, TN, '4.3', 'Style Factor Returns')

assign([
    'mktrf_cum_5d', 'mktrf_cum_20d',
    'smb_cum_5d', 'smb_cum_20d',
    'hml_cum_5d', 'hml_cum_20d',
    'umd_cum_5d', 'umd_cum_20d',
], T, TN, '4.4', 'Factor Momentum')

assign([
    'monthly_Mom6m', 'monthly_Mom12m', 'monthly_IntMom',
    'monthly_IndMom', 'monthly_MomVol',
    'monthly_ResidualMomentum', 'monthly_TrendFactor',
], T, TN, '4.5', 'Medium-Term Momentum')

assign([
    'monthly_REV6', 'monthly_MRreversal', 'monthly_LRreversal', 'monthly_High52',
], T, TN, '4.6', 'Reversal & Anchoring')

assign([
    'monthly_MomSeason', 'monthly_MomSeasonShort',
    'monthly_MomSeason06YrPlus', 'monthly_MomSeason11YrPlus', 'monthly_MomSeason16YrPlus',
], T, TN, '4.7', 'Seasonal Momentum')

assign([
    'monthly_Mom12mOffSeason', 'monthly_MomOffSeason',
    'monthly_MomOffSeason06YrPlus', 'monthly_MomOffSeason11YrPlus', 'monthly_MomOffSeason16YrPlus',
], T, TN, '4.8', 'Off-Season Momentum')

assign([
    'monthly_mom12m_chg_1m', 'monthly_mom6m_chg_1m', 'monthly_high52_chg_1m',
], T, TN, '4.9', 'Momentum Dynamics')

# ── THEME 5: Valuation ───────────────────────────────────────────────────────

T, TN = 5, "Valuation"

assign([
    'monthly_EP', 'monthly_EBM', 'monthly_BPEBM', 'monthly_EntMult',
], T, TN, '5.1', 'Earnings & Enterprise Multiples')

assign([
    'monthly_BMdec', 'monthly_AM',
    'monthly_IntanBM', 'monthly_IntanCFP', 'monthly_IntanEP', 'monthly_IntanSP',
], T, TN, '5.2', 'Book Value & Intangible-Adjusted')

assign([
    'monthly_SP', 'monthly_CF', 'monthly_cfp',
    'monthly_rev_yield', 'monthly_rev_yield_3m_avg',
], T, TN, '5.3', 'Sales, Cash Flow & Revenue Yields')

assign([
    'monthly_DivYieldST', 'monthly_NetPayoutYield', 'monthly_EquityDuration',
], T, TN, '5.4', 'Payout & Duration')

assign([
    'monthly_implied_return', 'monthly_ptg_median_implied',
    'monthly_ptg_upside', 'monthly_ptg_downside',
    'monthly_implied_return_3m_avg', 'monthly_implied_return_chg_1m',
], T, TN, '5.5', 'Forward Valuation from Analysts')

# ── THEME 6: Profitability & Earnings Quality ────────────────────────────────

T, TN = 6, "Profitability & Earnings Quality"

assign([
    'monthly_GP', 'monthly_OperProf', 'monthly_CBOperProf',
    'monthly_RoE', 'monthly_roaq', 'monthly_OPLeverage', 'monthly_AOP',
], T, TN, '6.1', 'Profitability Measures')

assign([
    'monthly_AbnormalAccruals', 'monthly_Accruals',
    'monthly_PctAcc', 'monthly_PctTotAcc', 'monthly_TotalAccruals',
    'monthly_NOA', 'monthly_dNoa', 'monthly_GrLTNOA',
], T, TN, '6.2', 'Accruals, Earnings Quality & NOA')

assign([
    'monthly_EarningsSurprise', 'monthly_AnnouncementReturn',
    'monthly_RevenueSurprise', 'monthly_NumEarnIncrease',
    'monthly_earnings_surprise_chg_1m', 'monthly_PredictedFE',
], T, TN, '6.3', 'Earnings Dynamics')

assign([
    'monthly_Tax', 'monthly_ChTax', 'monthly_ExclExp',
], T, TN, '6.4', 'Tax & Special Items')

# ── THEME 7: Investment & Corporate Structure ─────────────────────────────────

T, TN = 7, "Investment & Corporate Structure"

assign([
    'monthly_AssetGrowth', 'monthly_Investment', 'monthly_InvestPPEInv',
    'monthly_ChInv', 'monthly_ChInvIA',
    'monthly_grcapx', 'monthly_grcapx3y',
], T, TN, '7.1', 'Asset Growth & Capex')

assign([
    'monthly_ShareIss1Y', 'monthly_ShareIss5Y', 'monthly_CompEquIss',
    'monthly_ShareRepurchase', 'monthly_NetEquityFinance',
], T, TN, '7.2', 'Equity Issuance & Buybacks')

assign([
    'monthly_Leverage', 'monthly_BookLeverage',
    'monthly_DebtIssuance', 'monthly_CompositeDebtIssuance',
    'monthly_ConvDebt', 'monthly_NetDebtFinance', 'monthly_XFIN',
], T, TN, '7.3', 'Debt & Leverage')

assign([
    'monthly_ChEQ', 'monthly_ChNWC', 'monthly_ChNNCOA',
    'monthly_DelCOA', 'monthly_DelCOL', 'monthly_DelEqu',
    'monthly_DelFINL', 'monthly_DelLTI',
], T, TN, '7.4', 'Balance Sheet Changes')

assign([
    'monthly_DelNetFin', 'monthly_Cash', 'monthly_CashProd',
], T, TN, '7.5', 'Financial Assets & Cash')

assign([
    'monthly_ChAssetTurnover', 'monthly_GrSaleToGrInv', 'monthly_GrSaleToGrOverhead',
    'monthly_hire', 'monthly_MeanRankRevGrowth', 'monthly_RDS',
], T, TN, '7.6', 'Growth & Efficiency')

assign([
    'monthly_FirmAge', 'monthly_Herf', 'monthly_HerfAsset', 'monthly_HerfBE',
], T, TN, '7.7', 'Industry Structure & Firm Characteristics')

assign([
    'monthly_CredRatDG', 'monthly_DivInit', 'monthly_DivOmit',
    'monthly_DivSeason', 'monthly_IndIPO', 'monthly_Spinoff',
], T, TN, '7.8', 'Corporate Events')

# ── THEME 8: Analyst Expectations & Sentiment ─────────────────────────────────

T, TN = 8, "Analyst Expectations & Sentiment"

assign([
    'monthly_ptg_numest', 'monthly_ptg_dispersion', 'monthly_ptg_range',
    'monthly_ptg_upside_skew', 'monthly_ptg_numest_chg',
    'monthly_ptg_implied_range', 'monthly_ptg_implied_asymmetry',
], T, TN, '8.1', 'Price Target Consensus & Disagreement')

assign([
    'monthly_ptg_revision', 'monthly_ptg_revision_3m',
    'monthly_ptg_revision_accel', 'monthly_ptg_dispersion_chg_1m',
], T, TN, '8.2', 'Price Target Revisions')

assign([
    'monthly_rec_mean', 'monthly_rec_median', 'monthly_rec_numrec',
    'monthly_rec_buy_pct', 'monthly_rec_sell_pct',
    'monthly_rec_buy_sell_spread', 'monthly_rec_dispersion',
], T, TN, '8.3', 'Recommendation Consensus')

assign([
    'monthly_rec_revision', 'monthly_rec_revision_3m',
    'monthly_rec_upgrades', 'monthly_rec_downgrades',
    'monthly_rec_changes', 'monthly_rec_breadth',
], T, TN, '8.4', 'Recommendation Dynamics')

assign([
    'monthly_rec_revision_accel', 'monthly_rec_mean_chg_1m',
    'monthly_rec_dispersion_chg_1m', 'monthly_rec_mean_3m_avg',
], T, TN, '8.5', 'Recommendation Smoothed & Acceleration')

assign([
    'monthly_rev_numest', 'monthly_rev_numest_chg',
    'monthly_rev_dispersion', 'monthly_rev_range',
    'monthly_rev_revision_1m', 'monthly_rev_revision_3m',
    'monthly_rev_revision_accel',
], T, TN, '8.6', 'Revenue Estimates & Revisions')

assign([
    'monthly_rev_fy2_revision_1m', 'monthly_rev_fy2_dispersion',
    'monthly_rev_q_dispersion', 'monthly_rev_eps_divergence',
], T, TN, '8.7', 'Revenue Cross-Horizon & Divergence')

assign([
    'monthly_analyst_alignment', 'monthly_ptg_rev_alignment', 'monthly_fgr5yrLag',
], T, TN, '8.8', 'Cross-Signal Alignment & Growth Forecast')

assign([
    'monthly_ShortInterest', 'monthly_short_interest_chg_1m', 'monthly_short_interest_3m_avg',
    'monthly_DelBreadth', 'monthly_delbreadth_chg_1m',
], T, TN, '8.9', 'Short Interest & Institutional Positioning')

assign([
    'bullish', 'neutral', 'bearish', 'bullish_8w_ma', 'bull_bear_spread',
], T, TN, '8.10', 'Retail Sentiment Survey')

# ── THEME 9: Interest Rates & Monetary Policy ────────────────────────────────

T, TN = 9, "Interest Rates & Monetary Policy"

assign([
    'yield_1m', 'yield_3m', 'yield_6m', 'yield_1y', 'yield_2y',
    'yield_3y', 'yield_5y', 'yield_7y', 'yield_10y', 'yield_20y', 'yield_30y',
], T, TN, '9.1', 'Yield Curve Level')

assign([
    'slope_2y10y', 'slope_3m10y', 'slope_2y30y', 'slope_1y10y',
    'curve_2_5_10', 'curve_2_10_30',
], T, TN, '9.2', 'Yield Curve Shape')

assign([
    'tips_5y', 'tips_7y', 'tips_10y', 'tips_20y',
    'breakeven_5y', 'breakeven_10y',
    'real_rate_5y', 'real_rate_10y',
], T, TN, '9.3', 'Inflation Expectations & Real Rates')

assign([
    'fed_funds_eff', 'prime_rate', 'discount_rate', 'rf',
    'ff_2y_spread', 'ff_10y_spread',
], T, TN, '9.4', 'Policy Rates')

assign([
    'yield_2y_chg_1d', 'yield_2y_chg_5d',
    'yield_10y_chg_1d', 'yield_10y_chg_5d',
    'yield_30y_chg_1d', 'yield_30y_chg_5d',
    'slope_2y10y_chg_1d', 'slope_2y10y_chg_5d',
    'slope_3m10y_chg_1d', 'slope_3m10y_chg_5d',
    'breakeven_10y_chg_1d', 'breakeven_10y_chg_5d',
], T, TN, '9.5', 'Yield Dynamics')

assign([
    'yield_10y_vs_ma20', 'yield_10y_vol_20d',
], T, TN, '9.6', 'Yield Regime')

assign([
    'fed_assets', 'tga', 'reserves', 'bank_credit', 'ci_loans',
], T, TN, '9.7', 'Fed Balance Sheet & Banking')

assign([
    'monthly_b30ret', 'monthly_b30ind', 'monthly_b20ret', 'monthly_b20ind',
    'monthly_b10ret', 'monthly_b10ind', 'monthly_b7ret', 'monthly_b7ind',
    'monthly_b5ret', 'monthly_b5ind', 'monthly_b2ret', 'monthly_b2ind',
    'monthly_b1ret', 'monthly_b1ind', 'monthly_t90ret', 'monthly_t90ind',
], T, TN, '9.8', 'Bond Returns')

assign([
    'monthly_t30ret', 'monthly_t30ind',
    'monthly_b10ret_cum_3m', 'monthly_b30ret_cum_3m',
], T, TN, '9.9', 'T-Bill Returns & Bond Momentum')

assign([
    'monthly_bond_30y_2y_spread_ret', 'monthly_bond_10y_tbill_spread_ret',
    'monthly_bond_30y_10y_spread_ret', 'monthly_bond_5y_2y_spread_ret',
], T, TN, '9.10', 'Bond Term Premium')

assign([
    'monthly_m1', 'monthly_m2', 'monthly_monetary_base',
    'monthly_m1_mom', 'monthly_m2_mom', 'monthly_monetary_base_mom',
], T, TN, '9.11', 'Money Supply')

# ── THEME 10: Credit Conditions ───────────────────────────────────────────────

T, TN = 10, "Credit Conditions"

assign([
    'ig_oas', 'aaa_oas', 'aa_oas', 'a_oas', 'bbb_oas', 'moody_aaa',
], T, TN, '10.1', 'Investment Grade Spreads')

assign([
    'hy_oas', 'bb_oas', 'b_oas', 'ccc_oas', 'moody_baa',
], T, TN, '10.2', 'High Yield Spreads')

assign([
    'bbb_aaa_spread', 'bb_bbb_spread', 'moody_baa_aaa',
], T, TN, '10.3', 'Credit Quality Differentials')

assign([
    'hy_oas_chg_1d', 'hy_oas_chg_5d',
    'ig_oas_chg_1d', 'ig_oas_chg_5d',
    'bbb_aaa_chg_1d', 'bbb_aaa_chg_5d',
    'hy_oas_vs_ma20', 'hy_oas_vol_20d',
], T, TN, '10.4', 'Credit Dynamics & Volatility')

# ── THEME 11: Cross-Sectional Risk Profile ────────────────────────────────────

T, TN = 11, "Cross-Sectional Risk Profile"

assign([
    'monthly_Beta', 'monthly_BetaFP', 'monthly_BetaTailRisk',
    'monthly_BetaLiquidityPS', 'monthly_betaVIX',
    'monthly_beta_chg_1m', 'monthly_beta_chg_3m',
], T, TN, '11.1', 'Systematic Risk Exposure & Dynamics')

assign([
    'monthly_IdioVol3F', 'monthly_IdioVolAHT', 'monthly_VolSD',
    'monthly_idiovol_chg_1m', 'monthly_idiovol_3m_avg',
], T, TN, '11.2', 'Idiosyncratic Volatility')

assign([
    'monthly_RealizedVol', 'monthly_MaxRet', 'monthly_VarCF',
    'monthly_VolMkt', 'monthly_realvol_chg_1m',
], T, TN, '11.3', 'Realized Vol & Cash Flow Risk')

assign([
    'monthly_CoskewACX', 'monthly_Coskewness',
    'monthly_ReturnSkew', 'monthly_ReturnSkew3F',
], T, TN, '11.4', 'Coskewness & Tail Risk')

# ── THEME 12: Macroeconomic Fundamentals ──────────────────────────────────────

T, TN = 12, "Macroeconomic Fundamentals"

assign([
    'monthly_nonfarm_payrolls', 'monthly_unrate', 'monthly_u6_rate',
    'monthly_participation', 'monthly_avg_hourly_earnings', 'monthly_avg_weekly_hours',
    'monthly_jolts_openings', 'monthly_jolts_quits',
    'monthly_nonfarm_payrolls_mom', 'monthly_nfp_accel', 'monthly_nfp_mom_3m',
    'initial_claims', 'continued_claims',
], T, TN, '12.1', 'Employment & Labour')

assign([
    'monthly_cpi_urban', 'monthly_cpi_core', 'monthly_cpi_food',
    'monthly_cpi_energy', 'monthly_cpi_shelter', 'monthly_cpi_services',
    'monthly_pce_price', 'monthly_pce_core', 'monthly_import_prices',
    'monthly_cpi_urban_mom', 'monthly_cpi_core_mom',
    'monthly_pce_price_mom', 'monthly_pce_core_mom',
    'monthly_cpi_urban_yoy', 'monthly_cpi_core_yoy',
    'monthly_pce_price_yoy', 'monthly_pce_core_yoy',
], T, TN, '12.2', 'Inflation')

assign([
    'monthly_cpi_core_accel', 'monthly_pce_core_accel',
    'monthly_cpi_core_mom_3m',
    'monthly_cpiret', 'monthly_cpiind',
], T, TN, '12.3', 'Inflation Dynamics')

assign([
    'monthly_indpro', 'monthly_cap_util', 'monthly_manuf_prod',
    'monthly_indpro_mom', 'monthly_indpro_accel', 'monthly_indpro_mom_3m',
], T, TN, '12.4', 'Production & Manufacturing')

assign([
    'monthly_durable_orders', 'monthly_durable_ex_transport', 'monthly_factory_orders',
    'monthly_business_inventories', 'monthly_inv_sales_ratio',
    'monthly_durable_orders_mom', 'monthly_durable_ex_transport_mom', 'monthly_factory_orders_mom',
    'monthly_durable_orders_accel',
    'monthly_durable_orders_mom_3m', 'monthly_factory_orders_mom_3m',
], T, TN, '12.5', 'Orders & Inventories')

assign([
    'monthly_retail_sales', 'monthly_retail_ex_auto',
    'monthly_personal_income', 'monthly_personal_spending',
    'monthly_saving_rate', 'monthly_consumer_credit',
    'monthly_umich_sentiment', 'monthly_vehicle_sales',
    'monthly_retail_sales_mom', 'monthly_retail_ex_auto_mom',
    'monthly_personal_income_mom', 'monthly_personal_spending_mom',
    'monthly_consumer_credit_mom', 'monthly_retail_accel',
], T, TN, '12.6', 'Consumer & Spending')

assign([
    'monthly_spending_accel', 'monthly_income_accel',
    'monthly_retail_sales_mom_3m', 'monthly_spending_mom_3m',
], T, TN, '12.7', 'Consumer & Spending Dynamics')

assign([
    'monthly_housing_starts', 'monthly_building_permits',
    'monthly_new_home_sales', 'monthly_case_shiller',
    'monthly_housing_starts_mom', 'monthly_building_permits_mom',
    'monthly_new_home_sales_mom', 'monthly_case_shiller_mom',
    'monthly_housing_starts_accel', 'monthly_housing_starts_mom_3m',
    'monthly_case_shiller_yoy', 'monthly_new_home_sales_mom_3m',
], T, TN, '12.8', 'Housing')

assign([
    'monthly_trade_balance', 'monthly_exports', 'monthly_imports',
    'monthly_exports_mom', 'monthly_imports_mom',
    'monthly_trade_balance_12m_avg',
    'monthly_exports_accel', 'monthly_imports_accel', 'monthly_exports_mom_3m',
], T, TN, '12.9', 'Trade & External')

assign([
    'monthly_ism_manuf', 'monthly_ism_new_orders',
    'monthly_philly_fed', 'monthly_empire_state',
    'monthly_kansas_fed', 'monthly_chicago_fed_nai',
], T, TN, '12.10', 'Surveys & Leading Indicators')

# ── THEME 13: Global Markets, FX & Commodities ───────────────────────────────

T, TN = 13, "Global Markets, FX & Commodities"

assign([
    'fx_eur', 'fx_gbp', 'fx_jpy', 'fx_chf', 'fx_cad', 'fx_aud', 'fx_nok',
    'fx_eur_ret_1d', 'fx_jpy_ret_1d', 'fx_gbp_ret_1d', 'fx_aud_ret_1d',
], T, TN, '13.1', 'Developed Market FX')

assign([
    'fx_cny', 'fx_krw', 'fx_brl', 'fx_mxn', 'fx_cny_ret_1d',
], T, TN, '13.2', 'Emerging Market FX')

assign([
    'twexb', 'twexm', 'twexb_ret_1d', 'twexm_ret_1d',
], T, TN, '13.3', 'Dollar Indices')

assign([
    'widx_aus', 'widx_bra', 'widx_che', 'widx_chn',
    'widx_deu', 'widx_fra', 'widx_gbr', 'widx_hkg',
    'widx_ind', 'widx_jpn', 'widx_kor', 'widx_mex',
], T, TN, '13.4', 'World Equity Indices')

assign([
    'widx_avg_ret', 'widx_avg_cum_5d', 'widx_dispersion',
], T, TN, '13.5', 'Global Equity Dynamics')

assign([
    'wti_oil', 'brent_oil', 'natgas', 'brent_wti_spread',
    'wti_oil_ret_1d', 'wti_oil_ret_5d',
    'brent_oil_ret_1d', 'brent_oil_ret_5d',
    'natgas_ret_1d', 'natgas_ret_5d',
], T, TN, '13.6', 'Energy Commodities')

assign([
    'monthly_copper_monthly', 'monthly_aluminum_monthly',
    'monthly_wheat_monthly', 'monthly_corn_monthly', 'monthly_lumber_monthly',
    'monthly_copper_monthly_mom',
], T, TN, '13.7', 'Monthly Commodities')

assign([
    'stock_bond_corr_20d', 'risk_appetite',
], T, TN, '13.8', 'Cross-Asset Regime')

# ── THEME 14: Calendar & Regime ───────────────────────────────────────────────

T, TN = 14, "Calendar & Regime"

assign([
    'day_of_week', 'is_monday', 'is_friday',
    'month_of_year', 'is_quarter_end',
    'trading_days_to_month_end', 'is_turn_of_month', 'is_opex_week',
], T, TN, '14.1', 'Calendar Features')

assign([
    'vix_above_20', 'vix_above_30',
    'curve_inverted_2y10y', 'curve_inverted_3m10y', 'credit_stress',
], T, TN, '14.2', 'Regime Indicators')


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: VALIDATE — EVERY FEATURE ASSIGNED EXACTLY ONCE, ZERO ORPHANS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("VALIDATION")
print("=" * 90)

# 3a. Check for features in data but not in mapping (orphans)
mapped_factors = set(mapping.keys())
data_factors = set(features)

orphans = data_factors - mapped_factors
extra = mapped_factors - data_factors

print(f"\n  Features in data:    {len(data_factors)}")
print(f"  Features in mapping: {len(mapped_factors)}")
print(f"  Orphans (in data but not mapped): {len(orphans)}")
if orphans:
    for f in sorted(orphans):
        print(f"    ⚠ ORPHAN: {f}")

print(f"  Extras (in mapping but not in data): {len(extra)}")
if extra:
    for f in sorted(extra):
        print(f"    ⚠ EXTRA: {f}")

# 3b. Check for duplicates in mapping
factor_counts = Counter()
for f in mapping:
    factor_counts[f] += 1
dupes = {f: c for f, c in factor_counts.items() if c > 1}
if dupes:
    print(f"\n  ⚠ DUPLICATES: {len(dupes)}")
    for f, c in dupes.items():
        print(f"    {f}: assigned {c} times")
else:
    print(f"  ✓ No duplicate assignments")

# 3c. Summary by theme
print(f"\n  Theme summary:")
theme_counts = {}
subtheme_counts = {}
for f, (tid, tname, sid, sname) in mapping.items():
    if f in data_factors:  # only count factors actually in data
        theme_counts[tid] = theme_counts.get(tid, 0) + 1
        subtheme_counts[sid] = subtheme_counts.get(sid, 0) + 1

print(f"  {'#':<4s} {'Theme':<40s} {'Factors':>8s} {'Subthemes':>10s}")
print("  " + "-" * 65)
for tid in sorted(set(t[0] for t in mapping.values())):
    tname = [v[1] for v in mapping.values() if v[0] == tid][0]
    n_factors = theme_counts.get(tid, 0)
    n_sub = len([s for s, (t, _, _, _) in
                 {sid: next(v for v in mapping.values() if v[2] == sid)
                  for sid in subtheme_counts if sid.startswith(f"{tid}.")}.items()])
    # Count subthemes properly
    theme_subs = set()
    for f, (t, _, s, _) in mapping.items():
        if t == tid and f in data_factors:
            theme_subs.add(s)
    print(f"  {tid:<4d} {tname:<40s} {n_factors:>8d} {len(theme_subs):>10d}")

total_mapped = sum(theme_counts.values())
total_subs = len(set(v[2] for f, v in mapping.items() if f in data_factors))
print(f"\n  Total factors mapped: {total_mapped}")
print(f"  Total subthemes: {total_subs}")

if len(orphans) == 0 and len(extra) == 0 and len(dupes) == 0:
    print(f"\n  ✓ VALIDATION PASSED: {total_mapped} features assigned to {total_subs} subthemes with zero orphans")
else:
    print(f"\n  ⚠ VALIDATION FAILED — fix issues above before saving")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: BUILD AND SAVE CSV
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("SAVING")
print("=" * 90)

# Load the existing combined means inventory for source/description metadata
inv_path = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready/combined_means_factor_inventory.csv')
inv = pd.read_csv(inv_path)
inv_map = inv.set_index('column').to_dict('index')

rows = []
for f in features:
    if f in mapping:
        tid, tname, sid, sname = mapping[f]
        meta = inv_map.get(f, {})
        rows.append({
            'column': f,
            'theme_id': tid,
            'theme_name': tname,
            'subtheme_id': sid,
            'subtheme_name': sname,
            'base_factor': meta.get('base_factor', f),
            'frequency': meta.get('frequency', ''),
            'panel': meta.get('panel', ''),
            'source': meta.get('source', ''),
            'description': meta.get('description', ''),
        })

result = pd.DataFrame(rows)

out_path = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready/themes/combined_means_theme_assignment.csv')
result.to_csv(out_path, index=False)

print(f"\n  ✓ Saved: {out_path}")
print(f"    {len(result)} rows × {len(result.columns)} columns")
print(f"\n  Sample rows:")
print(result[['column', 'theme_id', 'theme_name', 'subtheme_id', 'subtheme_name']].head(10).to_string(index=False))

Features in combined means table: 722

VALIDATION

  Features in data:    722
  Features in mapping: 722
  Orphans (in data but not mapped): 0
  Extras (in mapping but not in data): 0
  ✓ No duplicate assignments

  Theme summary:
  #    Theme                                     Factors  Subthemes
  -----------------------------------------------------------------
  1    Liquidity & Market Quality                     63         10
  2    Order Flow & Participation                     70         10
  3    Volatility & Options                          110         12
  4    Momentum & Reversal                            47          9
  5    Valuation                                      24          5
  6    Profitability & Earnings Quality               24          4
  7    Investment & Corporate Structure               46          8
  8    Analyst Expectations & Sentiment               52         10
  9    Interest Rates & Monetary Policy               80         11
  10   Credit Conditi